## 2. MITRE ATT&CK integration 

This notebook extracts attack techniques, mitigations and the relationships between them from MITRE ATT&CK, so they can be linked to the CISA exercise scenarios.

We use the following datasets:

**Enterprise ATT&CK** | Contains: attack techniques targeting IT systems, such as phishing, ransomware and stolen accounts, together with possible mitigations.

**ICS ATT&CK** | Contains: attack techniques targeting industrial control systems, including effects on safety and production, together with possible mitigations. This dataset is needed for our CISA industrial control systems scenarios.

We do not include Mobile ATT&CK because our chosen scenarios do not involve attacks on mobile devices.

#### 1. Setting up
The libraries are imported and the project's data folder is located. A relative path is used so the notebook can run on different computers, from either the notebooks folder or the project root.

In [1]:
import json
import pandas as pd
from pathlib import Path

# looks for the data folder when running from the notebooks folder
data_dir = Path("../data")

# uses the data folder in the current directory if ../data does not exist
if not data_dir.exists():
    data_dir = Path("data")

# displays the full path to the selected data folder
print("Data folder:", data_dir.resolve())

Data folder: C:\Users\liisiparve\Projects\cybersecurity-project\data


#### 2. Getting the ATT&CK files
The Enterprise and ICS ATT&CK datasets are downloaded from the team's GitHub repository and saved in `data/raw/`.

In [5]:
# imports the library for downloading files
import requests

# defines the repository address
ATTACK_REPO = "https://raw.githubusercontent.com/Mastercoder-1000/attack-stix-data/master"
# and the ATT&CK datasets to use
domains = ["enterprise-attack", "ics-attack"]

# defines raw data folder path
raw_dir = data_dir / "raw"

# downloads each dataset if the local file is missing
for d in domains:
    path = raw_dir / f"{d}.json"
    
    if not path.exists():
        url = f"{ATTACK_REPO}/{d}/{d}.json"
        print(f"Downloading {url} ...")

        # downloads the file, raises error if the request fails
        response = requests.get(url, timeout=120)
        response.raise_for_status()

        # saves the downloaded file in the raw data folder
        path.write_bytes(response.content)

    # displays dataset name and file size in megabytes
    print(f"{d}: {path.stat().st_size / 1_000_000:.1f} MB")

enterprise-attack: 55.3 MB
ics-attack: 5.4 MB


#### 3. Helper functions
Functions are defined to extract ATT&CK IDs and tactic names, and to check which records are still active.

In [6]:
# 1. gets the ATT&CK ID
def get_mitre_id(refs):
    # returns None if the references are missing or not stored as a list
    if not isinstance(refs, list):
        return None

    # finds the MITRE ATT&CK reference and returns its ID
    for ref in refs:
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id")
    return None

# 2. gets the tactic names
def get_tactics(phases):
    # returns None if the phases are missing or not stored as a list
    if not isinstance(phases, list):
        return None

    # combines the tactic names into one string separated with commas
    return ", ".join(p["phase_name"] for p in phases)

# 3. checks which records are active
def is_active(df):
    # reads the revoked and deprecated flags, missing values treated as False
    revoked = df.get("revoked", pd.Series(False, index=df.index)).fillna(False)
    deprecated = df.get("x_mitre_deprecated", pd.Series(False, index=df.index)).fillna(False)

    # marks records as active if neither flag is True
    return (revoked != True) & (deprecated != True)

#### 4. Extracting techniques, mitigations and their links
A function that reads one ATT&CK dataset and returns three tables: techniques, mitigations, and links between them.

In [4]:
def extract_domain(domain):
    # Reads the JSON file and converts its objects into a table
    with open(raw_dir / f"{domain}.json", encoding="utf-8") as f:
        objects = pd.DataFrame(json.load(f)["objects"])

    # selects active attack techniques
    techniques = objects[(objects["type"] == "attack-pattern") & is_active(objects)].copy()
    # extracts the ATT&CK IDs and tactic names for each technique
    techniques["technique_id"] = techniques["external_references"].apply(get_mitre_id)
    techniques["tactics"] = techniques["kill_chain_phases"].apply(get_tactics)
    # keeps the required technique columns
    techniques = techniques[["id", "technique_id", "name", "description", "tactics"]]

    # selects active mitigations and extracts their ATT&CK IDs
    mitigations = objects[(objects["type"] == "course-of-action") & is_active(objects)].copy()
    mitigations["mitigation_id"] = mitigations["external_references"].apply(get_mitre_id)
    # keeps the required mitigation columns
    mitigations = mitigations[["id", "mitigation_id", "name", "description"]]

    # selects active links describing which mitigations address which techniques
    rel = objects[
        (objects["type"] == "relationship")
        & (objects["relationship_type"] == "mitigates")
        & is_active(objects)
    ]
    # keeps links whose mitigation and technique are both in the selected tables
    rel = rel[rel["source_ref"].isin(mitigations["id"]) & rel["target_ref"].isin(techniques["id"])]
    # keeps the linking IDs and removes repeated pairs
    rel = rel[["source_ref", "target_ref"]].drop_duplicates()

    # adds readable mitigation and technique IDs and names to each link
    technique_mitigations = (
        rel.merge(mitigations[["id", "mitigation_id", "name"]],
                  left_on="source_ref", right_on="id", validate="many_to_one")
           .rename(columns={"name": "mitigation_name"}).drop(columns="id")
           .merge(techniques[["id", "technique_id", "name"]],
                  left_on="target_ref", right_on="id", validate="many_to_one")
           .rename(columns={"name": "technique_name"})
           [["technique_id", "technique_name", "mitigation_id", "mitigation_name"]]
    )

    for df in (techniques, mitigations, technique_mitigations):
        df.insert(0, "domain", domain)

    return techniques, mitigations, technique_mitigations

#### 5. Extracting all domains
The function is applied to each domain and the results are combined into three tables.

*Why:* one combined table per object type is easier to join with the CISA data later than separate tables per domain. The per-domain counts are printed so they can be compared with the previous version.

In [5]:
all_tech, all_mit, all_rel = [], [], []

for d in domains:
    t, m, r = extract_domain(d)
    all_tech.append(t)
    all_mit.append(m)
    all_rel.append(r)
    print(f"{d}: {len(t)} techniques, {len(m)} mitigations, {len(r)} technique-mitigation links")

techniques = pd.concat(all_tech, ignore_index=True)
mitigations = pd.concat(all_mit, ignore_index=True)
technique_mitigations = pd.concat(all_rel, ignore_index=True)

print("\nTotal:", len(techniques), "techniques,", len(mitigations), "mitigations,",
      len(technique_mitigations), "links")

enterprise-attack: 706 techniques, 44 mitigations, 1477 technique-mitigation links
ics-attack: 104 techniques, 53 mitigations, 424 technique-mitigation links

Total: 810 techniques, 97 mitigations, 1901 links


#### 6. Validating the data
The extracted IDs and relationships are checked for missing values and duplicates.

*Why:* duplicates are now checked **within each domain** (`domain` + ID) instead of across the whole table. Each ATT&CK domain is a separate dataset, so the correct rule is that an ID must be unique inside its own domain.

In [6]:
assert techniques["technique_id"].notna().all(), "Missing technique IDs"
assert mitigations["mitigation_id"].notna().all(), "Missing mitigation IDs"
assert not techniques.duplicated(["domain", "technique_id"]).any(), "Duplicate technique IDs"
assert not mitigations.duplicated(["domain", "mitigation_id"]).any(), "Duplicate mitigation IDs"
assert technique_mitigations.notna().all().all(), "Missing values in relationships"

print("All validation checks passed.")
display(techniques.groupby("domain").size().rename("techniques").to_frame()
        .join(mitigations.groupby("domain").size().rename("mitigations"))
        .join(technique_mitigations.groupby("domain").size().rename("links")))

All validation checks passed.


,techniques,mitigations,links
domain,,,
enterprise-attack,706,44,1477
ics-attack,104,53,424


#### 7. Examples
Mitigations for one Enterprise technique and one ICS technique.

*Why:* a quick sanity check that the relationships make sense. Phishing (T1566) is used in several CISA scenarios; Loss of Safety (T0880) is relevant for the Industrial Control Systems scenario.

In [7]:
display(technique_mitigations[technique_mitigations["technique_id"] == "T1566"])
display(technique_mitigations[technique_mitigations["technique_id"] == "T0880"])

,domain,technique_id,technique_name,mitigation_id,mitigation_name
98,enterprise-attack,T1566,Phishing,M1047,Audit
651,enterprise-attack,T1566,Phishing,M1031,Network Intrusion Prevention
681,enterprise-attack,T1566,Phishing,M1054,Software Configuration
936,enterprise-attack,T1566,Phishing,M1021,Restrict Web-Based Content
1353,enterprise-attack,T1566,Phishing,M1049,Antivirus/Antimalware
1365,enterprise-attack,T1566,Phishing,M1017,User Training


,domain,technique_id,technique_name,mitigation_id,mitigation_name
1505,ics-attack,T0880,Loss of Safety,M0812,Safety Instrumented Systems
1535,ics-attack,T0880,Loss of Safety,M0805,Mechanical Protection Layers


#### 8. Saving the datasets
The combined tables are saved for the CISA-MITRE mapping.

*Why:* the file names are the same as before, so the other notebooks keep working. The only difference in the files is the extra `domain` column.

In [8]:
processed_dir = data_dir / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

techniques.to_csv(processed_dir / "mitre_techniques.csv", index=False, encoding="utf-8-sig")
mitigations.to_csv(processed_dir / "mitre_mitigations.csv", index=False, encoding="utf-8-sig")
technique_mitigations.to_csv(processed_dir / "technique_mitigations.csv", index=False, encoding="utf-8-sig")

print("MITRE datasets saved to", processed_dir)

MITRE datasets saved to ..\data\processed
